# AWS EC2 + MACE MD Workshop

This notebook is meant to run **locally on your laptop** in a regular Jupyter environment.

We will use Python/IPython kernel, but most practical steps are executed with shell cells using `%%bash`, plus a few optional Python cells for quick checks.

## What we will do
1. Confirm the local environment.
2. Check AWS identity and instance metadata.
3. Connect to the EC2 instance with SSH.
4. Install the remote Python environment.
5. Upload MACE model files and geometries.
6. Run a small MD simulation with a MACE-based ASE workflow.
7. Monitor Linux-side and AWS-side resource usage.
8. Periodically download logs and trajectories for local inspection.
9. Choose whether to stop the instance, preserve the volume, or terminate everything.

## Important storage decision
If you may want to reuse the installed environment after the workshop, set **Delete on termination = No** for the root EBS volume during instance launch.

That way, you can terminate the EC2 instance later but keep the volume and reattach or reuse it.

## For this to work you need

- Python 3.10+
- JupyterLab or Jupyter Notebook
- AWS CLI v2
- OpenSSH (`ssh`, `scp`)
- A code editor such as VS Code or PyCharm or Jupyter server
- Python packages in the install.py

If you need Jupyter locally, one simple route is:
```bash
python3 -m venv workshop-nb
source workshop-nb/bin/activate
pip install --upgrade pip jupyterlab ase matplotlib
jupyter lab
```

In [ ]:
%%bash # indicates that this notebook cell is executed in bash, not Python
set -e 
which aws # Check if AWS CLI is installed
aws --version
which ssh # Check if SSH is installed
ssh -V
which scp # Check if SCP is installed
scp -V
python3 --version
pwd

/opt/homebrew/bin/aws
aws-cli/2.34.3 Python/3.13.12 Darwin/25.3.0 source/arm64
/usr/bin/ssh


OpenSSH_10.2p1, LibreSSL 3.3.6


/usr/bin/scp
Python 3.12.2
/Users/akolganov/PycharmProjects/ISE_CloudCompute_Workshop


## Fill in your connection details

Edit the next cell before running the notebook.

In [ ]:
REGION = "eu-central-1" ## Change to your region
INSTANCE_ID = "i-xxxxxxxxxxxxxxxxx" ## Change to your instance id
PUBLIC_IP = "x.x.x.x" ## Change to your instance public IP
KEY_PATH = "~/.ssh/workshop.pem" ## Change to your private key path
SSH_USER = "ubuntu" ## Change to your instance username, e.g. "ec2-user" for Amazon Linux, "ubuntu" for Ubuntu, etc.
LOCAL_MODEL = "./models/mace.model" ## Change to your local model path, e.g. "./models/mace.model"
LOCAL_GEOMETRY = "./inputs/structure.xyz" ## Change to your local geometry path, e.g. "./inputs/structure.xyz"
LOCAL_WORKDIR = "./workshop_local" ## Change to your local workdir, e.g. "./workshop_local"
REMOTE_WORKDIR = "/home/ubuntu/workshop_remote" ## Change to your remote workdir, e.g. "/home/ubuntu/workshop_remote"

In [ ]:
from pathlib import Path
import os

KEY_PATH_EXPANDED = os.path.expanduser(KEY_PATH) # Expand the key path to handle the "~" symbol
Path(LOCAL_WORKDIR).mkdir(parents=True, exist_ok=True) # Create the local workdir if it doesn't exist
print("Local workdir:", Path(LOCAL_WORKDIR).resolve())
print("Expanded key path:", KEY_PATH_EXPANDED)

## Check AWS login and region

This verifies that the AWS CLI is configured locally and that you are authenticated in the expected account.

In [ ]:
%%bash
set -e
aws sts get-caller-identity # Check if AWS credentials are configured correctly and can access STS
aws configure list # Show the current AWS CLI configuration, including region and credentials source
aws ec2 describe-instances --instance-ids $INSTANCE_ID --region $REGION # Check if we can access the EC2 instance information using the provided instance ID and region

## Inspect the EC2 instance

This confirms the instance state, public IP, availability zone, VPC, subnet, and attached security groups.

In [ ]:
%%bash
set -e
REGION="eu-central-1"
INSTANCE_ID="i-xxxxxxxxxxxxxxxxx"

"""This command retrieves detailed information about a specific EC2 instance using the AWS CLI."""  

aws ec2 describe-instances   --region "$REGION"   
--instance-ids "$INSTANCE_ID"
--query 'Reservations[*].Instances[*].{ID:InstanceId,State:State.Name,Type:InstanceType,IP:PublicIpAddress,AZ:Placement.AvailabilityZone,VPC:VpcId,Subnet:SubnetId,SG:SecurityGroups[*].GroupId}'  
--output table

### If you prefer, replace the hard-coded values above

For a cleaner workflow, manually edit the shell cells to match the variables you set earlier.

Notebook shell cells do not automatically inherit Python variables unless you pass them explicitly.

## First SSH connection test

This checks that your key permissions are correct and that you can reach the remote host.

In [ ]:
%%bash
set -e
KEY_PATH="$HOME/.ssh/workshop.pem" ## Change to your full key path
PUBLIC_IP="x.x.x.x" ## Change to your instance public IP
SSH_USER="ubuntu" ## Change to your instance username, e.g. "ec2-user" for Amazon Linux, "ubuntu" for Ubuntu, etc.

chmod 400 "$KEY_PATH" # Set the correct permissions for the private key

"""This command attempts to establish an SSH connection to the EC2 instance using the provided key, username, and public IP address. 
It also runs some basic commands on the remote instance to verify the connection and gather system information. 
Make sure to replace the placeholders with your actual values before running this command.
"""

ssh -o StrictHostKeyChecking=accept-new -i "$KEY_PATH" ${SSH_USER}@${PUBLIC_IP} 'hostname; uname -a; uptime'

## Remote bootstrap

We will now adjust the virtual enviromnemt. Luckily for us, AWS deep learning AMI has already the environemnt with a lot of stuff - we need just to install a bit libraries

In [ ]:
%%bash
set -e
KEY_PATH="$HOME/.ssh/workshop.pem"
PUBLIC_IP="x.x.x.x"
SSH_USER="ubuntu"

ssh -i "$KEY_PATH" ${SSH_USER}@${PUBLIC_IP} <<'EOF'
set -e
sudo apt update # Update package lists
sudo apt install -y git tmux htop
source /opt/pytorch/bin/activate
pip install --upgrade pip
pip install ase matplotlib
pip install mace-torch
python3 - <<'PY'
import sys
print(sys.version)
import ase, torch
print('ASE OK')
print('Torch OK')
PY
EOF

## Prepare a simple MD driver locally

This cell writes a small ASE + MACE MD script **on your laptop**.

We'll do it for rather small testing and before uploading it to the AWS

In [ ]:
from pathlib import Path
import textwrap

scripts_dir = Path(LOCAL_WORKDIR) / "scripts"
scripts_dir.mkdir(parents=True, exist_ok=True)
run_md = scripts_dir / "run_md.py"


run_md.write_text(textwrap.dedent('''
from ase import io
from ase.md.langevin import Langevin
from ase import units
from ase.io.trajectory import Trajectory
from mace.calculators import MACECalculator
import csv

atoms = io.read('structure.xyz')
calc = MACECalculator(model_paths='mace.model', device='cpu', default_dtype='float32')
atoms.calc = calc

dyn = Langevin(atoms, timestep=0.5 * units.fs, temperature_K=300.0, friction=0.02)
traj = Trajectory('md.traj', 'w', atoms)
logf = open('md_energy.log', 'w')
writer = csv.writer(open('thermo.csv', 'w', newline=''))
writer.writerow(['step', 'potential_energy_eV'])

step_counter = {'i': 0}

def write_frame():
    e = atoms.get_potential_energy()
    i = step_counter['i']
    traj.write(atoms)
    logf.write(f"{i} {e:.8f}\n")
    logf.flush()
    writer.writerow([i, e])
    step_counter['i'] += 1

dyn.attach(write_frame, interval=10)
write_frame()
dyn.run(500)
io.write('final.xyz', atoms)
logf.close()
''').strip() + "\n")
print(run_md)

## Upload model, geometry, and MD script

Make sure these local files exist before running:
- your MACE model file, here assumed to be `./models/mace.model`
- your input geometry, here assumed to be `./inputs/structure.xyz`
- the generated `run_md.py` script from the previous cell

In [ ]:
%%bash
set -e
KEY_PATH="$HOME/.ssh/workshop.pem"
PUBLIC_IP="x.x.x.x"
SSH_USER="ubuntu"

"""scp commands to transfer the files to the remote EC2 instance."""

scp -i "$KEY_PATH" ./models/mace.model ${SSH_USER}@${PUBLIC_IP}:~/mace.model
scp -i "$KEY_PATH" ./inputs/structure.xyz ${SSH_USER}@${PUBLIC_IP}:~/structure.xyz
scp -i "$KEY_PATH" ./workshop_local/scripts/run_md.py ${SSH_USER}@${PUBLIC_IP}:~/run_md.py

## Start the MD job in tmux

This launches the simulation in a detached `tmux` session so that it keeps running even if your SSH connection closes.

In [ ]:
%%bash
set -e
KEY_PATH="$HOME/.ssh/workshop.pem"
PUBLIC_IP="x.x.x.x"
SSH_USER="ubuntu"


ssh -i "$KEY_PATH" ${SSH_USER}@${PUBLIC_IP} <<'EOF'
set -e
source ~/mace-env/bin/activate
tmux new-session -d -s mdjob 'python3 ~/run_md.py > md.stdout 2>&1'
tmux ls
EOF

## Quick job-status check

This confirms that the process is present and prints the latest log lines.

In [ ]:
%%bash
set -e
KEY_PATH="$HOME/.ssh/workshop.pem"
PUBLIC_IP="x.x.x.x"
SSH_USER="ubuntu"

ssh -i "$KEY_PATH" ${SSH_USER}@${PUBLIC_IP} <<'EOF'
set -e
ps aux | grep run_md.py | grep -v grep || true
tail -n 20 ~/md.stdout || true
ls -lh ~/md.traj ~/md_energy.log ~/thermo.csv ~/final.xyz 2>/dev/null || true
EOF

## Linux-side resource monitoring

Run this cell repeatedly during the job.

Since we are on a GPU instance, use `nvidia-smi` command to report device utilization and memory usage.

In [ ]:
%%bash
set -e
KEY_PATH="$HOME/.ssh/workshop.pem"
PUBLIC_IP="x.x.x.x"
SSH_USER="ubuntu"

ssh -i "$KEY_PATH" ${SSH_USER}@${PUBLIC_IP} <<'EOF'
set -e
echo '=== uptime ==='
uptime
echo '=== memory ==='
free -h
echo '=== disk ==='
df -h
echo '=== cpu ==='
top -bn1 | head -n 20
echo '=== gpu ==='
nvidia-smi || true
EOF

## Periodically download logs and trajectory files

This is the easiest way for students to inspect progress locally.

In [ ]:
%%bash
set -e
KEY_PATH="$HOME/.ssh/workshop.pem"
PUBLIC_IP="x.x.x.x"
SSH_USER="ubuntu"

mkdir -p ./workshop_local/downloads
scp -i "$KEY_PATH" ${SSH_USER}@${PUBLIC_IP}:~/md.stdout ./workshop_local/downloads/ || true
scp -i "$KEY_PATH" ${SSH_USER}@${PUBLIC_IP}:~/md_energy.log ./workshop_local/downloads/ || true
scp -i "$KEY_PATH" ${SSH_USER}@${PUBLIC_IP}:~/thermo.csv ./workshop_local/downloads/ || true
scp -i "$KEY_PATH" ${SSH_USER}@${PUBLIC_IP}:~/md.traj ./workshop_local/downloads/ || true
scp -i "$KEY_PATH" ${SSH_USER}@${PUBLIC_IP}:~/final.xyz ./workshop_local/downloads/ || true
ls -lh ./workshop_local/downloads

## Optional local inspection

This cell checks whether the downloaded files are present.

If ASE is installed locally, you can extend this to visualize the trajectory.

In [ ]:
from pathlib import Path

for name in ["md.stdout", "md_energy.log", "thermo.csv", "md.traj", "final.xyz"]:
    p = Path("./workshop_local/downloads") / name
    print(f"{name}: {'present' if p.exists() else 'missing'}")

## Optional: preserve the root volume for later reuse

If the root EBS volume was launched with **Delete on termination = No**, then you can terminate the instance later and keep the volume.

This is useful if you want to preserve the installed software stack and your output files for future workshops or additional calculations.

In [ ]:
%%bash
set -e
REGION="eu-central-1"
INSTANCE_ID="i-xxxxxxxxxxxxxxxxx"

aws ec2 describe-instances   --region "$REGION"   --instance-ids "$INSTANCE_ID"   --query 'Reservations[0].Instances[0].BlockDeviceMappings[*].{Device:DeviceName,Volume:Ebs.VolumeId,DeleteOnTermination:Ebs.DeleteOnTermination}'   --output table

## End-of-workshop decision

Choose one path:

### Option A: Stop the instance
Use this if you want to resume later on the same instance.

### Option B: Terminate the instance, keep the volume
Use this if the volume was configured to survive termination.

### Option C: Terminate everything
Use this if you are fully done and do not need the environment anymore.

In [ ]:
%%bash
set -e
REGION="eu-central-1"
INSTANCE_ID="i-xxxxxxxxxxxxxxxxx"

aws ec2 stop-instances --region "$REGION" --instance-ids "$INSTANCE_ID"

In [ ]:
%%bash
set -e
REGION="eu-central-1"
INSTANCE_ID="i-xxxxxxxxxxxxxxxxx"

aws ec2 terminate-instances --region "$REGION" --instance-ids "$INSTANCE_ID"

## Final reminders

- Stopped instances do not consume compute, but attached storage still costs money.
- Preserved volumes also continue to cost money until deleted.
- Download important results before terminating anything.
- Keep your SSH private key safe and never share it.